# CellPert — putting predictions on a common output scale

CellPert is trained on LINCS L1000 and evaluated on Mini Tahoe, and the two sit on
different output scales: the LINCS training matrix has mean 8.43, the Mini Tahoe targets
have mean 0.14. Metrics react to that gap very differently. Pearson and Spearman are
invariant to any positive affine transformation of the prediction and are unaffected.
Mean squared error, R^2, cosine similarity and SSIM are not invariant, and on raw output
they largely measure the scale gap rather than the prediction.

This notebook needs `output/predictions/plate_1_predictions.pkl`, which the end-to-end
tutorial writes. The first cell produces it if it is not already there.


In [1]:
import os
import subprocess

PKL = './output/predictions/plate_1_predictions.pkl'
if os.path.exists(PKL):
    print('predictions already present:', PKL)
else:
    print('running inference on plate 1 to produce them')
    subprocess.run(['python', 'src/main.py', '--test_flag', '--test_dataset_id', '1',
                    '--batch_size', '64'], check=True)
print('exists:', os.path.exists(PKL))


predictions already present: ./output/predictions/plate_1_predictions.pkl
exists: True


## The diagnostic

One scalar pair `(a, b)` is fitted by least squares on a single plate,

```
a = Cov(P, G) / Var(P)      b = mean(G) - a * mean(P)
```

then frozen and applied unchanged to every other plate as `a * P + b`. Two degrees of
freedom in total, so it cannot encode anything perturbation-specific; it only places the
output on the target scale.

Two checks belong with it. Pearson and Spearman must come out identical before and
after, which doubles as a correctness check on the implementation. And the fitted `a`
must be reported: if `a` is near zero the calibrated prediction collapses to the constant
`b`, which equals the mean of the ground truth, and every error-based metric then
describes the ground truth alone rather than the model.

On the released checkpoint the fit gives `a = 0.657512` and `b = -4.083115`, and over the
held-out plates the metrics move as follows.

| metric | raw | calibrated |
| --- | --- | --- |
| Pearson | 0.3868 | 0.3868 |
| Spearman | 0.1380 | 0.1380 |
| MSE | 40.9121 | 0.1142 |
| R^2 | -555.63 | -0.0963 |
| Cosine similarity | 0.3825 | 0.4967 |
| SSIM | 0.0117 | 0.1934 |


In [2]:
import pickle
import numpy as np

# Predictions written by src/main.py in --test_flag mode, or by src/run_all.sh.
with open('./output/predictions/plate_1_predictions.pkl', 'rb') as f:
    d = pickle.load(f)
P = np.asarray(d['predictions'], dtype=np.float64)    # cells x 965
G = np.asarray(d['ground_truth'], dtype=np.float64)

# Fit the two scalars on this plate, then treat them as fixed.
p, g = P.ravel(), G.ravel()
a = float(np.cov(p, g, bias=True)[0, 1] / p.var())
b = float(g.mean() - a * p.mean())
print('fitted on plate 1:  a = %.6f   b = %.6f' % (a, b))
if abs(a) < 1e-3:
    print('warning: a is near zero, the calibrated output is essentially the constant b')

Q = a * P + b        # apply to this plate, and to every other plate unchanged


def report(name, X):
    xc = X - X.mean(1, keepdims=True)
    gc = G - G.mean(1, keepdims=True)
    den = np.linalg.norm(xc, axis=1) * np.linalg.norm(gc, axis=1)
    pearson = np.nanmean(np.where(den > 0, (xc * gc).sum(1) / np.maximum(den, 1e-12), np.nan))
    cosine = np.nanmean((X * G).sum(1) /
                        (np.linalg.norm(X, axis=1) * np.linalg.norm(G, axis=1) + 1e-12))
    mse = float(((X - G) ** 2).mean())
    r2 = float(1 - ((X - G) ** 2).sum() / ((G - G.mean()) ** 2).sum())
    print('%-12s MSE %10.4f   R2 %10.4f   Pearson %.4f   cosine %.4f'
          % (name, mse, r2, pearson, cosine))


report('raw', P)
report('calibrated', Q)

# Pearson is identical on both lines. That is the point: the affine changes the scale
# of the output and nothing about the ordering or the shape of what the model predicts.


fitted on plate 1:  a = 0.653283   b = -4.055497
raw          MSE    39.9033   R2  -203.6400   Pearson 0.3485   cosine 0.4078
calibrated   MSE     0.1837   R2     0.0578   Pearson 0.3485   cosine 0.5062
